# 03 - Feature Engineering
## Customer-Level RFM & Behavioral Aggregation
**Objective:** Transform session-level transactions into customer-level analytical records featuring Recency, Frequency, Monetary (RFM), engagement metrics, and cart abandonment behavior.


In [ ]:
import os
import pandas as pd
import numpy as np

clean_path = os.path.join("..", "data", "processed", "cleaned_data.csv")
df = pd.read_csv(clean_path)
df['visit_date_dt'] = pd.to_datetime(df['visit_date_dt'])


### 1. Customer-Level Aggregation


In [ ]:
max_date = df['visit_date_dt'].max()

customer_df = df.groupby('customer_id').agg(
    total_sessions=('session_id', 'count'),
    total_purchases=('purchased', 'sum'),
    total_revenue=('revenue', 'sum'),
    total_cart_adds=('added_to_cart', 'sum'),
    total_cart_abandons=('cart_abandoned', 'sum'),
    avg_pages_viewed=('pages_viewed', 'mean'),
    avg_time_on_site_sec=('time_on_site_sec', 'mean'),
    avg_discount_received=('discount_percent', 'mean'),
    avg_rating_given=('rating', 'mean'),
    last_visit_date=('visit_date_dt', 'max'),
    location=('location', lambda x: x.mode().iloc[0] if not x.empty else 0),
    primary_device=('device_type', lambda x: x.mode().iloc[0] if not x.empty else 0),
    primary_category=('product_category', lambda x: x.mode().iloc[0] if not x.empty else 0)
).reset_index()


### 2. Engineering Derived Metrics


In [ ]:
# Recency in days
customer_df['recency_days'] = (max_date - customer_df['last_visit_date']).dt.days

# Average Order Value (AOV)
customer_df['avg_order_value'] = np.where(
    customer_df['total_purchases'] > 0,
    customer_df['total_revenue'] / customer_df['total_purchases'],
    0.0
)

# Cart Abandonment Rate
customer_df['cart_abandonment_rate'] = np.where(
    customer_df['total_cart_adds'] > 0,
    customer_df['total_cart_abandons'] / customer_df['total_cart_adds'],
    0.0
)

# Purchase Conversion Rate
customer_df['purchase_conversion_rate'] = customer_df['total_purchases'] / customer_df['total_sessions']


### 3. Feature Distribution Inspection


In [ ]:
customer_df[['total_sessions', 'total_purchases', 'total_revenue', 'avg_order_value', 'cart_abandonment_rate', 'recency_days']].describe()


### 4. Save Customer Features Table


In [ ]:
out_path = os.path.join("..", "data", "processed", "customer_features.csv")
customer_df.to_csv(out_path, index=False)
print(f"Customer feature table saved: {customer_df.shape[0]:,} profiles -> {out_path}")


### Conclusion
Generated 8,442 unique customer behavioral vectors containing RFM, cart interaction rates, and engagement metrics ready for clustering and spending prediction.
